In [3]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
import random

# Cargar los modelos ya construidos
ranking_elo = pd.read_csv('../datos/procesados/ranking_elo.csv')
promedios   = pd.read_csv('../datos/procesados/promedios_goles.csv', index_col='seleccion')

# ✅ Columna correcta: 'puntos_elo', índice: 'seleccion'
elo = ranking_elo.set_index('seleccion')['puntos_elo'].to_dict()

# Promedios globales como fallback
MEDIA_GOL = promedios['prom_anotados'].mean()
print(f"Datos cargados. Media global de goles: {MEDIA_GOL:.2f}")
print(f"Equipos en ELO: {len(elo)}")
print(f"Equipos en promedios: {len(promedios)}")

# Verificación rápida
print(f"\nTop 5 ELO:")
top5 = ranking_elo.sort_values('puntos_elo', ascending=False).head(5)
print(top5[['seleccion','puntos_elo']].to_string(index=False))

Datos cargados. Media global de goles: 1.28
Equipos en ELO: 223
Equipos en promedios: 212

Top 5 ELO:
seleccion  puntos_elo
    Spain        1457
   France        1388
Argentina        1363
  England        1362
    Japan        1350


In [4]:

# 12 grupos de 4 equipos (Mundial 2026)
grupos = {
    'A': ['United States', 'Panama', 'Morocco', 'Senegal'],
    'B': ['Argentina', 'Peru', 'Australia', 'Slovenia'],
    'C': ['Spain', 'Japan', 'Serbia', 'Sudan'],
    'D': ['France', 'Nigeria', 'Belgium', 'Paraguay'],
    'E': ['Mexico', 'Venezuela', 'South Korea', 'Hungary'],
    'F': ['Brazil', 'Uruguay', 'Cameroon', 'DR Congo'],
    'G': ['England', 'Netherlands', 'Algeria', 'Ecuador'],
    'H': ['Portugal', 'Turkey', 'Egypt', 'Costa Rica'],
    'I': ['Germany', 'Poland', 'Ivory Coast', 'Indonesia'],
    'J': ['Croatia', 'Chile', 'Iran', 'Zimbabwe'],
    'K': ['Colombia', 'Switzerland', 'Saudi Arabia', 'Albania'],
    'L': ['Canada', 'Denmark', 'Austria', 'New Zealand'],
}

# Verifica nombres contra tus datos
print("=== Equipos NO encontrados en ELO ===")
faltantes_elo = []
for grupo, equipos in grupos.items():
    for e in equipos:
        if e not in elo:
            faltantes_elo.append(e)
            print(f"  ⚠ Grupo {grupo}: '{e}'")

if not faltantes_elo:
    print("  ✅ Todos los equipos encontrados en ELO")

print("\n=== Equipos NO encontrados en promedios ===")
faltantes_prom = []
for grupo, equipos in grupos.items():
    for e in equipos:
        if e not in promedios.index:
            faltantes_prom.append(e)
            print(f"  ⚠ Grupo {grupo}: '{e}'")

if not faltantes_prom:
    print("  ✅ Todos los equipos encontrados en promedios")

# Ver cómo se llaman algunos equipos en tus datos (para comparar)
print("\n=== Muestra de nombres en tus datos ===")
nombres = list(elo.keys())
buscar = ['Korea', 'Congo', 'Ivory', 'United', 'South']
for b in buscar:
    matches = [n for n in nombres if b.lower() in n.lower()]
    if matches:
        print(f"  '{b}' → {matches}")

=== Equipos NO encontrados en ELO ===
  ✅ Todos los equipos encontrados en ELO

=== Equipos NO encontrados en promedios ===
  ✅ Todos los equipos encontrados en promedios

=== Muestra de nombres en tus datos ===
  'Korea' → ['South Korea', 'North Korea']
  'Congo' → ['DR Congo', 'Congo']
  'Ivory' → ['Ivory Coast']
  'United' → ['United States', 'United Arab Emirates', 'United States Virgin Islands']
  'South' → ['South Korea', 'South Africa', 'South Sudan']


In [6]:
# 12 grupos de 4 equipos (Mundial 2026 - formato real)
grupos = {
    'A': ['United States', 'Panama', 'Morocco', 'Senegal'],
    'B': ['Argentina', 'Peru', 'Australia', 'Slovenia'],
    'C': ['Spain', 'Japan', 'Serbia', 'Sudan'],
    'D': ['France', 'Nigeria', 'Belgium', 'Paraguay'],
    'E': ['Mexico', 'Venezuela', 'South Korea', 'Hungary'],
    'F': ['Brazil', 'Uruguay', 'Cameroon', 'DR Congo'],
    'G': ['England', 'Netherlands', 'Algeria', 'Ecuador'],
    'H': ['Portugal', 'Turkey', 'Egypt', 'Costa Rica'],
    'I': ['Germany', 'Poland', 'Ivory Coast', 'Indonesia'],
    'J': ['Croatia', 'Chile', 'Iran', 'Zimbabwe'],
    'K': ['Colombia', 'Switzerland', 'Saudi Arabia', 'Albania'],
    'L': ['Canada', 'Denmark', 'Austria', 'New Zealand'],
}

# Verifica cuántos equipos tienes en tus datos
for grupo, equipos in grupos.items():
    for e in equipos:
        if e not in elo:
            print(f"  ⚠ '{e}' no encontrado en ELO — ajusta el nombre")
print("Verificación completa.")

Verificación completa.


In [7]:
def goles_esperados(equipo1, equipo2):
    """Calcula lambda (goles esperados) usando Poisson + ajuste ELO."""
    at1 = promedios.loc[equipo1, 'prom_anotados'] if equipo1 in promedios.index else MEDIA_GOL
    df1 = promedios.loc[equipo1, 'prom_recibidos'] if equipo1 in promedios.index else MEDIA_GOL
    at2 = promedios.loc[equipo2, 'prom_anotados'] if equipo2 in promedios.index else MEDIA_GOL
    df2 = promedios.loc[equipo2, 'prom_recibidos'] if equipo2 in promedios.index else MEDIA_GOL

    # Lambda base: ataque equipo1 vs defensa equipo2
    lam1 = (at1 * df2) / MEDIA_GOL
    lam2 = (at2 * df1) / MEDIA_GOL

    # Ajuste ELO: el más fuerte anota un poco más
    elo1 = elo.get(equipo1, 1000)
    elo2 = elo.get(equipo2, 1000)
    diff = (elo1 - elo2) / 400
    factor = 10 ** diff / (1 + 10 ** diff)  # entre 0 y 1
    ajuste = 0.3 * (factor - 0.5)           # entre -0.15 y +0.15

    lam1 = max(0.1, lam1 + ajuste)
    lam2 = max(0.1, lam2 - ajuste)
    return lam1, lam2


def simular_partido(equipo1, equipo2, penales=False):
    """Simula un partido. Si penales=True, nunca hay empate."""
    lam1, lam2 = goles_esperados(equipo1, equipo2)
    g1 = poisson.rvs(lam1)
    g2 = poisson.rvs(lam2)

    if penales and g1 == g2:
        # Penales: 50/50 ajustado por ELO
        prob_gana1 = 1 / (1 + 10 ** ((elo.get(equipo2,1000) - elo.get(equipo1,1000)) / 400))
        ganador = equipo1 if random.random() < prob_gana1 else equipo2
        return g1, g2, ganador, True  # True = fue a penales

    ganador = equipo1 if g1 > g2 else (equipo2 if g2 > g1 else None)
    return g1, g2, ganador, False

In [8]:
def simular_grupo(nombre_grupo, equipos):
    """Juega los 6 partidos del grupo y devuelve la tabla."""
    tabla = {e: {'pts':0, 'gf':0, 'gc':0, 'dg':0} for e in equipos}

    for i in range(len(equipos)):
        for j in range(i+1, len(equipos)):
            e1, e2 = equipos[i], equipos[j]
            g1, g2, ganador, _ = simular_partido(e1, e2)

            tabla[e1]['gf'] += g1; tabla[e1]['gc'] += g2
            tabla[e2]['gf'] += g2; tabla[e2]['gc'] += g1

            if ganador == e1:
                tabla[e1]['pts'] += 3
            elif ganador == e2:
                tabla[e2]['pts'] += 3
            else:
                tabla[e1]['pts'] += 1; tabla[e2]['pts'] += 1

    for e in tabla:
        tabla[e]['dg'] = tabla[e]['gf'] - tabla[e]['gc']

    df = pd.DataFrame(tabla).T.sort_values(
        ['pts', 'dg', 'gf'], ascending=False
    )
    return df


# Simular todos los grupos
clasificados = {}
print("=== FASE DE GRUPOS ===")
for nombre, equipos in grupos.items():
    tabla = simular_grupo(nombre, equipos)
    clasificados[nombre] = tabla.index.tolist()[:2]  # Top 2
    print(f"\nGrupo {nombre}:")
    print(tabla[['pts','gf','gc','dg']].to_string())
    print(f"  → Clasifican: {clasificados[nombre][0]} y {clasificados[nombre][1]}")

=== FASE DE GRUPOS ===

Grupo A:
               pts  gf  gc  dg
United States    7   8   2   6
Senegal          5   3   1   2
Morocco          4   5   5   0
Panama           0   2  10  -8
  → Clasifican: United States y Senegal

Grupo B:
           pts  gf  gc  dg
Australia    9   7   2   5
Peru         6   3   4  -1
Argentina    3   3   2   1
Slovenia     0   1   6  -5
  → Clasifican: Australia y Peru

Grupo C:
        pts  gf  gc  dg
Spain     6   9   5   4
Serbia    6   8   7   1
Japan     6   6   6   0
Sudan     0   0   5  -5
  → Clasifican: Spain y Serbia

Grupo D:
          pts  gf  gc  dg
Belgium     7   8   1   7
Nigeria     7   3   0   3
France      3   4   4   0
Paraguay    0   1  11 -10
  → Clasifican: Belgium y Nigeria

Grupo E:
             pts  gf  gc  dg
Hungary        6   6   3   3
South Korea    6   4   4   0
Mexico         3   5   3   2
Venezuela      3   1   6  -5
  → Clasifican: Hungary y South Korea

Grupo F:
          pts  gf  gc  dg
Brazil      9   9   1   8
Came

In [9]:
def ronda_eliminatoria(nombre_ronda, partidos):
    """Juega una ronda. partidos = lista de tuplas (equipo1, equipo2)."""
    print(f"\n{'='*40}")
    print(f"  {nombre_ronda}")
    print(f"{'='*40}")
    ganadores = []
    for e1, e2 in partidos:
        g1, g2, ganador, penales = simular_partido(e1, e2, penales=True)
        sufijo = " (pen)" if penales else ""
        print(f"  {e1} {g1} - {g2} {e2}{sufijo}  →  {ganador}")
        ganadores.append(ganador)
    return ganadores


# --- Octavos de final ---
octavos = [
    (clasificados['A'][0], clasificados['B'][1]),
    (clasificados['C'][0], clasificados['D'][1]),
    (clasificados['E'][0], clasificados['F'][1]),
    (clasificados['G'][0], clasificados['H'][1]),
    (clasificados['I'][0], clasificados['J'][1]),
    (clasificados['K'][0], clasificados['L'][1]),
    (clasificados['B'][0], clasificados['A'][1]),
    (clasificados['D'][0], clasificados['C'][1]),
]
g_octavos = ronda_eliminatoria("OCTAVOS DE FINAL", octavos)

g_cuartos = ronda_eliminatoria("CUARTOS DE FINAL", [
    (g_octavos[0], g_octavos[1]),
    (g_octavos[2], g_octavos[3]),
    (g_octavos[4], g_octavos[5]),
    (g_octavos[6], g_octavos[7]),
])
g_semis = ronda_eliminatoria("SEMIFINALES", [
    (g_cuartos[0], g_cuartos[1]),
    (g_cuartos[2], g_cuartos[3]),
])
g_final = ronda_eliminatoria("GRAN FINAL", [
    (g_semis[0], g_semis[1]),
])

print(f"\n🏆 CAMPEÓN DEL MUNDIAL 2026: {g_final[0]} 🏆")


  OCTAVOS DE FINAL
  United States 0 - 0 Peru (pen)  →  United States
  Spain 1 - 1 Nigeria (pen)  →  Spain
  Hungary 0 - 1 Cameroon  →  Cameroon
  Algeria 0 - 0 Egypt (pen)  →  Egypt
  Germany 2 - 0 Chile  →  Germany
  Saudi Arabia 1 - 1 Austria (pen)  →  Austria
  Australia 0 - 0 Senegal (pen)  →  Senegal
  Belgium 0 - 1 Serbia  →  Serbia

  CUARTOS DE FINAL
  United States 0 - 1 Spain  →  Spain
  Cameroon 2 - 0 Egypt  →  Cameroon
  Germany 1 - 3 Austria  →  Austria
  Senegal 1 - 0 Serbia  →  Senegal

  SEMIFINALES
  Spain 1 - 1 Cameroon (pen)  →  Spain
  Austria 0 - 1 Senegal  →  Senegal

  GRAN FINAL
  Spain 1 - 1 Senegal (pen)  →  Spain

🏆 CAMPEÓN DEL MUNDIAL 2026: Spain 🏆


In [11]:
from collections import Counter
import sys, io

N = 1000
campeones = Counter()

for i in range(N):
    # Suprimir prints de simular_grupo
    clasif = {}
    for nombre, equipos in grupos.items():
        tabla = simular_grupo(nombre, equipos)
        clasif[nombre] = tabla.index.tolist()[:2]

    oc = [(clasif['A'][0],clasif['B'][1]),(clasif['C'][0],clasif['D'][1]),
          (clasif['E'][0],clasif['F'][1]),(clasif['G'][0],clasif['H'][1]),
          (clasif['I'][0],clasif['J'][1]),(clasif['K'][0],clasif['L'][1]),
          (clasif['B'][0],clasif['A'][1]),(clasif['D'][0],clasif['C'][1])]

    go = [simular_partido(a,b,penales=True)[2] for a,b in oc]
    gc = [simular_partido(go[i],go[i+1],penales=True)[2] for i in range(0,8,2)]
    gs = [simular_partido(gc[i],gc[i+1],penales=True)[2] for i in range(0,4,2)]
    gf = simular_partido(gs[0], gs[1], penales=True)[2]
    campeones[gf] += 1

    # Progreso cada 100 simulaciones
    if (i+1) % 100 == 0:
        print(f"  Simulación {i+1}/{N}...")

print(f"\n🎲 PROBABILIDADES DE CAMPEONATO ({N} simulaciones)\n")
for equipo, wins in campeones.most_common(15):
    pct = wins/N*100
    barra = '█' * int(pct/2)
    print(f"{equipo:20s} {barra:25s} {pct:.1f}%")

print("\n💡 Tip: cambia N=5000 para resultados más estables")

  Simulación 100/1000...
  Simulación 200/1000...
  Simulación 300/1000...
  Simulación 400/1000...
  Simulación 500/1000...
  Simulación 600/1000...
  Simulación 700/1000...
  Simulación 800/1000...
  Simulación 900/1000...
  Simulación 1000/1000...

🎲 PROBABILIDADES DE CAMPEONATO (1000 simulaciones)

Japan                ██████                    12.0%
Spain                █████                     11.8%
Morocco              █████                     10.7%
England              ██                        5.4%
France               ██                        5.2%
Senegal              ██                        4.5%
South Korea          ██                        4.3%
United States        ██                        4.0%
Argentina            ██                        4.0%
Australia            ██                        4.0%
Belgium              █                         3.3%
Ivory Coast          █                         3.2%
Germany              █                         2.7%
Iran             